In [19]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import wfdb
import ast
import logging
from tqdm import tqdm
from scipy.stats import zscore
import torch

BASE_DIR = Path('./')
OUT_DIR = './PTBXL'
PTBXL_DIR = BASE_DIR / 'raw/PTBXL'
PTBXL_CLASSES = ["NORM", "MI", "STTC", "HYP", "CD"]

In [3]:
# load the dataset annotation and convert `scp_codes` values to list
metadata = pd.read_csv(PTBXL_DIR / 'ptbxl_database.csv', index_col='ecg_id')
metadata.scp_codes = metadata.scp_codes.apply(lambda x: ast.literal_eval(x))

print('samples: ', len(metadata))
metadata.head()

samples:  21799


,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr


In [4]:
# load the SCP statements' mapping
label_map_csv = pd.read_csv(PTBXL_DIR / "scp_statements.csv", index_col=0)
label_map = label_map_csv[label_map_csv.diagnostic == 1]  # keep only diagnostic
label_map.head()

,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass,Statement Category,SCP-ECG Statement Description,AHA code,aECG REFID,CDISC Code,DICOM Code
NDT,non-diagnostic T abnormalities,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,non-diagnostic T abnormalities,NaN,NaN,NaN,NaN
NST_,non-specific ST changes,1.0,1.0,NaN,STTC,NST_,Basic roots for coding ST-T changes and abnorm...,non-specific ST changes,145.0,MDC_ECG_RHY_STHILOST,NaN,NaN
DIG,digitalis-effect,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,suggests digitalis-effect,205.0,NaN,NaN,NaN
LNGQT,long QT-interval,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,long QT-interval,148.0,NaN,NaN,NaN
NORM,normal ECG,1.0,NaN,NaN,NORM,NORM,Normal/abnormal,normal ECG,1.0,NaN,NaN,F-000B7


In [5]:
train_split = metadata.query("strat_fold < 9").copy()
val_split = metadata.query("strat_fold == 9").copy()
test_split = metadata.query("strat_fold == 10").copy()

len(train_split), len(val_split), len(test_split)

(17418, 2183, 2198)

In [6]:
def read_records(split: pd.DataFrame, path_col = 'filename_hr'):
    data = []
    labels = []

    for _, record in tqdm(split.iterrows(), total=len(split), desc="Reading records", unit="record"):
        # read the signal
        signal, _ = wfdb.rdsamp(PTBXL_DIR / record[path_col])
        signal = torch.tensor(signal.T, dtype=torch.float32)  # shape (12, length)

        # extract the super classes
        classes = []
        for label in  record["scp_codes"].keys():
            if label in label_map.index:
                classes.append(label_map.loc[label].diagnostic_class)
        classes = list(set(classes))

        class_idx = [PTBXL_CLASSES.index(_class) for _class in classes]
        one_hot = np.zeros(len(PTBXL_CLASSES))
        one_hot[class_idx] = 1

        data.append(signal)
        labels.append(one_hot)

    return np.array(data), np.array(labels)

In [7]:
train_data, train_label = read_records(train_split)
val_data, val_label = read_records(val_split)
test_data, test_label = read_records(test_split)

Reading records: 100%|██████████| 2198/2198 [00:05<00:00, 434.07record/s]


In [16]:
#### Convert from multi-label to multi-class ####
train_mask_onelabel = np.sum(train_label, axis=1) == 1 # filter samples with only one class
X_train, y_train = train_data[train_mask_onelabel], train_label[train_mask_onelabel]
X_train = zscore(X_train, axis=2)
print("train shape: ", X_train.shape, y_train.shape)

val_mask_onelabel = np.sum(val_label, axis=1) == 1
X_val, y_val = val_data[val_mask_onelabel], val_label[val_mask_onelabel]
X_val = zscore(X_val, axis=2)
print("val shape: ", X_val.shape, y_val.shape)

test_mask_onelabel = np.sum(test_label, axis=1) == 1
X_test, y_test = test_data[test_mask_onelabel], test_label[test_mask_onelabel]
X_test = zscore(X_test, axis=2)
print("test shape: ", X_test.shape, y_test.shape)

train shape:  (12957, 12, 5000) (12957, 5)
val shape:  (1637, 12, 5000) (1637, 5)
test shape:  (1650, 12, 5000) (1650, 5)


In [20]:
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

np.save(os.path.join(OUT_DIR, 'X_train.npy'), X_train)
np.save(os.path.join(OUT_DIR, 'y_train.npy'), y_train)

np.save(os.path.join(OUT_DIR, 'X_val.npy'), X_val)
np.save(os.path.join(OUT_DIR, 'y_val.npy'), y_val)

np.save(os.path.join(OUT_DIR, 'X_test.npy'), X_test)
np.save(os.path.join(OUT_DIR, 'y_test.npy'), y_test)